<a href="https://colab.research.google.com/github/sadeeshDeSilva/STRATIA-defense-impact-analysis/blob/Economic_Forecasting_Model/Economics_LSTM_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Import Libraries:
import pandas as pd
import numpy as np

df = pd.read_csv('/Final_Master_Dataset.csv')
df.head()

,Country,Year,D_Expenditure_GDP,Conflict_Intensity,Health_Expenditure_(% of GDP),Education_Expenditure_(% of GDP),Environmental impact (CO2e/capita),Refugees,Asylum Seekers,Total number of deaths,Population,GDP_Growth_Annual,GDP_PerCapita_Growth_Annual,GDP_Current_USDollars,Inflation_Annual,Unemployment_Total_of_TLF,Unemployment_Male_of_TMF,Unemployment_Female_of_TFF
0,USA,1994,4.215265,NaN,NaN,NaN,19.83,364,0,0,263126000,4.029023,2.761109,7.290000e+12,2.607442,6.119,6.174,6.052
1,USA,1995,3.860246,NaN,NaN,NaN,19.79,243,0,0,266278000,2.684431,1.468929,7.640000e+12,2.805420,5.650,5.633,5.671
2,USA,1996,3.554982,NaN,NaN,NaN,20.14,141,0,0,269394000,3.772773,2.572464,8.070000e+12,2.931204,5.451,5.435,5.470
3,USA,1997,3.554982,NaN,NaN,NaN,20.90,75,0,0,272657000,4.447128,3.197166,8.580000e+12,2.337690,5.000,4.938,5.075
4,USA,1998,3.201558,NaN,NaN,NaN,20.83,50,0,0,275854000,4.483133,3.272230,9.060000e+12,1.552279,4.511,4.419,4.623


In [6]:
# Use only the required features for the model:
features = [
    "Country",
    "Year",
    "D_Expenditure_GDP",
    "Refugees",
    "Asylum Seekers",
    "Total number of deaths",
    "Population",
    "GDP_Growth_Annual",
    "GDP_PerCapita_Growth_Annual",
    "GDP_Current_USDollars",
    "Inflation_Annual",
    "Unemployment_Total_of_TLF",
    "Unemployment_Male_of_TMF",
    "Unemployment_Female_of_TFF"
]

df = df[features]
df.head()


,Country,Year,D_Expenditure_GDP,Refugees,Asylum Seekers,Total number of deaths,Population,GDP_Growth_Annual,GDP_PerCapita_Growth_Annual,GDP_Current_USDollars,Inflation_Annual,Unemployment_Total_of_TLF,Unemployment_Male_of_TMF,Unemployment_Female_of_TFF
0,USA,1994,4.215265,364,0,0,263126000,4.029023,2.761109,7.290000e+12,2.607442,6.119,6.174,6.052
1,USA,1995,3.860246,243,0,0,266278000,2.684431,1.468929,7.640000e+12,2.805420,5.650,5.633,5.671
2,USA,1996,3.554982,141,0,0,269394000,3.772773,2.572464,8.070000e+12,2.931204,5.451,5.435,5.470
3,USA,1997,3.554982,75,0,0,272657000,4.447128,3.197166,8.580000e+12,2.337690,5.000,4.938,5.075
4,USA,1998,3.201558,50,0,0,275854000,4.483133,3.272230,9.060000e+12,1.552279,4.511,4.419,4.623


In [7]:
# Standarize column names:
df.columns = [
    "Country",
    "Year",
    "Defense_GDP",
    "Refugees",
    "Asylum_Seekers",
    "Deaths",
    "Population",
    "GDP_Growth",
    "GDP_PC_Growth",
    "GDP_Current_USD",
    "Inflation",
    "Unemp_Total",
    "Unemp_Male",
    "Unemp_Female"
]

# Sort by country and year for LSTM modelling:
df = df.sort_values(["Country", "Year"])
df.reset_index(drop=True, inplace=True)
# Handle missing-values:
df = df.groupby("Country").apply(               # Country-wise interpolation..
    lambda x: x.interpolate(method="linear")
).reset_index(drop=True)

df = df.dropna().reset_index(drop=True)

# Verify year ranges:
df.groupby("Country")["Year"].agg(["min", "max"])  # (Min; 1994, Max; 2024)

# Split Train/Test by year:
train_df = df[df["Year"] <= 2020].copy()
test_df  = df[df["Year"] > 2020].copy()

print("Train years:", train_df["Year"].min(), "-", train_df["Year"].max())
print("Test years:", test_df["Year"].min(), "-", test_df["Year"].max())


Train years: 1994 - 2020
Test years: 2021 - 2024


/tmp/ipython-input-151388299.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  lambda x: x.interpolate(method="linear")
/tmp/ipython-input-151388299.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  lambda x: x.interpolate(method="linear")
/tmp/ipython-input-151388299.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  lambda x: x.interpolate(method="linear")
/tmp/ipython-input-151388299.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  lambda x: x.interpolate(method="linear")
/tmp

In [8]:
# Define Inputs vs Targets:
input_features = [
    "Defense_GDP",
    "Refugees",
    "Asylum_Seekers",
    "Deaths",
    "Population",
    "GDP_PC_Growth",
    "GDP_Current_USD",
    "Unemp_Male",
    "Unemp_Female"
]

target_features = [
    "GDP_Growth",
    "Inflation",
    "Unemp_Total"
]

# Final Sanity Check:
print("INPUT SHAPE:", train_df[input_features].shape , "\n\nINPUT FEATURES:" , train_df[input_features])
print("TARGET SHAPE:", train_df[target_features].shape , "\n\nTARGET FEATURES:" , train_df[target_features])



INPUT SHAPE: (135, 9) 

INPUT FEATURES:      Defense_GDP  Refugees  Asylum_Seekers  Deaths  Population  GDP_PC_Growth  \
0       1.693480    113908               0       0  1191835000      11.801890   
1       1.686234    104691               0       0  1204855000       9.832961   
2       1.652727    105806               0       0  1217550000       8.829514   
3       1.632651    106731               0       4  1230075000       8.174846   
4       1.655081    109396               0       0  1241935000       6.890190   
..           ...       ...             ...     ...         ...            ...   
146     3.402602       302             286       0   324353340       1.022666   
147     3.297724       310            2196       8   326608609       1.750141   
148     3.304001       332            2834       1   328529577       2.364442   
149     3.412158       297            2940       0   330226227       2.056766   
150     3.650514       349            2491       0   331577720      -